# Model Comparison -- Test Set Analysis

Companion notebook to `model_comparison.ipynb` (train it first). This notebook **loads the five already-trained
models** (no retraining) and runs a detailed test-set analysis on the *same test samples* used everywhere else in this
project (`models/data_split.json`), mirroring `notebooks/m11_unet/m11_unet_test_sample_analysis.ipynb` but extended to
all five candidates side by side:

1. Published U-Net (pretrained encoder) -- `models/best_model.pth`
2. U-Net, no pretrained weights -- `models/comparison/UNet_NoPretrained.pth`
3. U-Net++ (pretrained encoder) -- `models/comparison/UNetPlusPlus_Pretrained.pth`
4. DeepLabV3+ (pretrained encoder) -- `models/comparison/DeepLabV3Plus_Pretrained.pth`
5. Random Forest (classical ML) -- `models/comparison/RandomForest_Classical.joblib`

**Outputs go to a dedicated subfolder** `results/model_comparison/test_analysis/`:
- `per_sample_all_models.csv` -- every metric, every sample, every model
- `summary_table.csv` -- mean +/- std per class per model (same shape as `results/analysis_results/table1/2`)
- `best_median_worst_samples.csv` -- the samples selected for qualitative comparison
- `all_test_samples_comparison.pdf` -- one page per test sample, all 5 predictions side by side
- `sample_XX_<name>_comparison.pdf` / `.tiff` -- the same, one file per sample

**Prerequisite:** run `model_comparison.ipynb` first so `models/comparison/*.pth` and `*.joblib` exist.
**Run with the project's Python 3.10 / CUDA environment**, not the repo's `.venv`.


In [1]:
import json
from pathlib import Path
from typing import Dict, Optional, Tuple

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tv_models
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from scipy.ndimage import uniform_filter
from tqdm.auto import tqdm

import segmentation_models_pytorch as smp

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 9

print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")


c:\Users\user_picm\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch 2.5.1+cu121 | CUDA available: True


## 1. Configuration

In [2]:
class Config:
    DATA_SPLIT_PATH = Path("../../models/data_split.json")
    REFERENCE_MODEL_PATH = Path("../../models/best_model.pth")
    COMPARISON_DIR = Path("../../models/comparison")
    RESULTS_DIR = Path("../../results/model_comparison/test_analysis")

    ENCODER_NAME = 'resnet34'
    INPUT_SIZE = (512, 512)
    NUM_CLASSES = 4
    CLASS_NAMES = ['Background', 'Tissue', 'OS', 'Vaginal']
    CLASS_COLORS = {0: [0, 0, 0], 1: [0, 0, 255], 2: [0, 255, 0], 3: [255, 0, 0]}

    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


Config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)

with open(Config.DATA_SPLIT_PATH, 'r') as f:
    data_split = json.load(f)

print(f"Device: {Config.DEVICE}")
print(f"Test samples (same split used throughout the project): {len(data_split['test'])}")


Device: cuda
Test samples (same split used throughout the project): 12


## 2. Data Loading

Identical to `model_comparison.ipynb` so predictions/metrics are computed the same way.

In [3]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)


def _robust_normalize(arr: np.ndarray, lo_pct=1, hi_pct=99) -> np.ndarray:
    arr = np.nan_to_num(arr.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    lo, hi = np.percentile(arr, [lo_pct, hi_pct])
    if hi <= lo:
        return np.zeros_like(arr, dtype=np.float32)
    return np.clip((arr - lo) / (hi - lo), 0, 1).astype(np.float32)


def resize_2d(arr: np.ndarray, size: Tuple[int, int], is_mask: bool) -> np.ndarray:
    t = torch.from_numpy(arr).float().unsqueeze(0).unsqueeze(0)
    mode = 'nearest' if is_mask else 'bilinear'
    kwargs = {} if is_mask else {'align_corners': True}
    t = F.interpolate(t, size=size, mode=mode, **kwargs)
    return t.squeeze(0).squeeze(0).numpy()


def load_sample(npz_path: Path) -> Optional[Dict[str, np.ndarray]]:
    """Load M11, auxiliary polarimetric maps, and the combined 4-class mask at native resolution."""
    try:
        with np.load(npz_path, allow_pickle=True) as data:
            m11 = None
            if 'nM11s' in data:
                m11 = np.array(data['nM11s'], dtype=np.float32)
            elif 'M11s' in data:
                raw = np.array(data['M11s'], dtype=np.float32)
                m11 = (raw - raw.min()) / (raw.max() - raw.min() + 1e-6)
            if m11 is None or m11.ndim != 2:
                return None
            m11 = np.nan_to_num(m11, nan=0.0, posinf=0.0, neginf=0.0)

            tissue_mask = None
            for key in ['tissue_mask', 'annotation_mask']:
                if key in data and data[key].size > 0:
                    tissue_mask = np.array(data[key]) > 0
                    break
            if tissue_mask is None:
                return None

            os_mask = np.array(data['os_mask']) > 0 if ('os_mask' in data and data['os_mask'].size > 0) else None

            vag_mask = None
            for key in ['vaginal_mask', 'vaginal_wall', 'vaginal_wall_mask']:
                if key in data and data[key].size > 0:
                    vag_mask = np.array(data[key]) > 0
                    break

            combined_mask = np.zeros_like(tissue_mask, dtype=np.int64)
            combined_mask[tissue_mask] = 1
            if os_mask is not None:
                combined_mask[os_mask] = 2
            if vag_mask is not None:
                combined_mask[vag_mask] = 3

            aux = {}
            for out_key, npz_key in [('mdepol', 'Mdepols'), ('mdiatt', 'Mdiattenuations'),
                                      ('morient', 'Morientations'), ('linr', 'linrs')]:
                aux[out_key] = np.array(data[npz_key], dtype=np.float32) if npz_key in data else np.zeros_like(m11)

            return {'m11': m11, 'mask': combined_mask, **aux}
    except Exception as e:
        print(f"Error loading {npz_path}: {e}")
        return None


print("Data loading functions defined")


Data loading functions defined


## 3. Metrics

Same definitions as `model_comparison.ipynb` / `m11_unet_test_sample_analysis.ipynb`.

In [4]:
def class_confusion(pred: np.ndarray, gt: np.ndarray, class_id: int):
    p = (pred == class_id)
    g = (gt == class_id)
    tp = np.logical_and(p, g).sum(dtype=np.int64)
    fp = np.logical_and(p, ~g).sum(dtype=np.int64)
    fn = np.logical_and(~p, g).sum(dtype=np.int64)
    tn = np.logical_and(~p, ~g).sum(dtype=np.int64)
    return tp, fp, fn, tn


def per_class_metrics(pred: np.ndarray, gt: np.ndarray, class_id: int) -> Dict[str, float]:
    tp, fp, fn, tn = class_confusion(pred, gt, class_id)
    union = tp + fp + fn
    dice = 1.0 if union == 0 else (2.0 * tp) / (2 * tp + fp + fn)
    iou = 1.0 if union == 0 else tp / union
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return {'dice': dice, 'iou': iou, 'precision': precision, 'recall': recall,
            'f1': f1, 'specificity': specificity}


def evaluate_sample(pred: np.ndarray, gt: np.ndarray) -> Dict:
    per_class = {c: per_class_metrics(pred, gt, c) for c in range(Config.NUM_CLASSES)}
    pixel_acc = float((pred == gt).mean())
    tissue_dice = float(np.mean([per_class[c]['dice'] for c in range(1, Config.NUM_CLASSES)]))
    return {'per_class': per_class, 'pixel_accuracy': pixel_acc, 'mean_tissue_dice': tissue_dice}


print("Metric functions defined")


Metric functions defined


## 4. Load the Five Trained Models

Reconstructs each architecture with `encoder_weights=None` (faster / no re-download) and loads the trained weights from
disk -- the checkpoint already contains the full trained encoder, so the ImageNet init is irrelevant here.

In [5]:
class PublishedUNet(nn.Module):
    """Matches run_trained/model_utils.py / notebooks/m11_unet training architecture exactly."""

    def __init__(self, num_classes):
        super().__init__()
        self.encoder = tv_models.resnet34(weights=None)
        enc_layers = list(self.encoder.children())
        self.enc1 = nn.Sequential(*enc_layers[:3])
        self.enc2 = nn.Sequential(*enc_layers[3:5])
        self.enc3 = enc_layers[5]
        self.enc4 = enc_layers[6]
        self.enc5 = enc_layers[7]
        self.dec4 = self._block(512 + 256, 256)
        self.dec3 = self._block(256 + 128, 128)
        self.dec2 = self._block(128 + 64, 64)
        self.dec1 = self._block(64 + 64, 32)
        self.final = nn.Conv2d(32, num_classes, 1)

    def _block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU())

    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(e1); e3 = self.enc3(e2); e4 = self.enc4(e3); e5 = self.enc5(e4)
        d4 = F.interpolate(e5, size=e4.shape[2:], mode='bilinear')
        d4 = self.dec4(torch.cat([d4, e4], 1))
        d3 = F.interpolate(d4, size=e3.shape[2:], mode='bilinear')
        d3 = self.dec3(torch.cat([d3, e3], 1))
        d2 = F.interpolate(d3, size=e2.shape[2:], mode='bilinear')
        d2 = self.dec2(torch.cat([d2, e2], 1))
        d1 = F.interpolate(d2, size=e1.shape[2:], mode='bilinear')
        d1 = self.dec1(torch.cat([d1, e1], 1))
        out = F.interpolate(d1, scale_factor=2, mode='bilinear')
        return self.final(out)


def load_checkpoint_state(path: Path) -> dict:
    ckpt = torch.load(path, map_location=Config.DEVICE, weights_only=False)
    return ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt


models_dict = {}

reference_model = PublishedUNet(num_classes=Config.NUM_CLASSES).to(Config.DEVICE)
reference_model.load_state_dict(load_checkpoint_state(Config.REFERENCE_MODEL_PATH))
reference_model.eval()
models_dict['UNet_Pretrained_Published'] = reference_model

unet_scratch = smp.Unet(encoder_name=Config.ENCODER_NAME, encoder_weights=None,
                         in_channels=3, classes=Config.NUM_CLASSES).to(Config.DEVICE)
unet_scratch.load_state_dict(load_checkpoint_state(Config.COMPARISON_DIR / 'UNet_NoPretrained.pth'))
unet_scratch.eval()
models_dict['UNet_NoPretrained'] = unet_scratch

unet_pp = smp.UnetPlusPlus(encoder_name=Config.ENCODER_NAME, encoder_weights=None,
                            in_channels=3, classes=Config.NUM_CLASSES).to(Config.DEVICE)
unet_pp.load_state_dict(load_checkpoint_state(Config.COMPARISON_DIR / 'UNetPlusPlus_Pretrained.pth'))
unet_pp.eval()
models_dict['UNetPlusPlus_Pretrained'] = unet_pp

deeplab = smp.DeepLabV3Plus(encoder_name=Config.ENCODER_NAME, encoder_weights=None,
                              in_channels=3, classes=Config.NUM_CLASSES).to(Config.DEVICE)
deeplab.load_state_dict(load_checkpoint_state(Config.COMPARISON_DIR / 'DeepLabV3Plus_Pretrained.pth'))
deeplab.eval()
models_dict['DeepLabV3Plus_Pretrained'] = deeplab

rf_ckpt = joblib.load(Config.COMPARISON_DIR / 'RandomForest_Classical.joblib')
rf_model = rf_ckpt['model']
RF_FEATURE_NAMES = rf_ckpt['feature_names']
assert tuple(rf_ckpt['input_size']) == Config.INPUT_SIZE, "RF was trained at a different input size than Config.INPUT_SIZE"
models_dict['RandomForest_Classical'] = rf_model

MODEL_ORDER = ['UNet_Pretrained_Published', 'UNet_NoPretrained', 'UNetPlusPlus_Pretrained',
               'DeepLabV3Plus_Pretrained', 'RandomForest_Classical']
MODEL_LABELS = {'UNet_Pretrained_Published': 'U-Net (pretrained, published)',
                'UNet_NoPretrained': 'U-Net (no pretrained)',
                'UNetPlusPlus_Pretrained': 'U-Net++ (pretrained)',
                'DeepLabV3Plus_Pretrained': 'DeepLabV3+ (pretrained)',
                'RandomForest_Classical': 'Random Forest (classical)'}

print("Loaded 5 models:", MODEL_ORDER)


Loaded 5 models: ['UNet_Pretrained_Published', 'UNet_NoPretrained', 'UNetPlusPlus_Pretrained', 'DeepLabV3Plus_Pretrained', 'RandomForest_Classical']


## 5. Prediction Functions

One wrapper per model family, all returning `(pred_mask_native_res, gt_mask_native_res)` for a given `.npz` path.

In [6]:
def extract_rf_features(sample: Dict[str, np.ndarray], size=Config.INPUT_SIZE) -> np.ndarray:
    m11 = resize_2d(sample['m11'], size, is_mask=False)
    mdepol = resize_2d(_robust_normalize(sample['mdepol']), size, is_mask=False)
    mdiatt = resize_2d(_robust_normalize(sample['mdiatt']), size, is_mask=False)
    morient = resize_2d((sample['morient'] + np.pi / 2) / np.pi, size, is_mask=False)
    linr = resize_2d(np.nan_to_num(sample['linr']) / np.pi, size, is_mask=False)
    local_mean = uniform_filter(m11, size=5)
    local_std = np.sqrt(np.clip(uniform_filter(m11 ** 2, size=5) - local_mean ** 2, 0, None))
    feats = np.stack([m11, mdepol, mdiatt, morient, linr, local_mean, local_std], axis=-1)
    return feats.astype(np.float32)


@torch.no_grad()
def predict_deep(model: nn.Module, sample: Dict[str, np.ndarray]) -> np.ndarray:
    m11, gt = sample['m11'], sample['mask']
    x = resize_2d(m11, Config.INPUT_SIZE, is_mask=False)
    x_rgb = np.stack([x, x, x], axis=0)
    x_norm = (x_rgb - IMAGENET_MEAN) / IMAGENET_STD
    x_t = torch.from_numpy(x_norm).float().unsqueeze(0).to(Config.DEVICE)
    logits = model(x_t)
    pred = torch.argmax(logits, dim=1).float()
    pred = F.interpolate(pred.unsqueeze(1), size=gt.shape, mode='nearest').squeeze().long()
    return pred.cpu().numpy()


def predict_rf(sample: Dict[str, np.ndarray]) -> np.ndarray:
    feats = extract_rf_features(sample)
    h, w, n_feat = feats.shape
    pred_flat = rf_model.predict(feats.reshape(-1, n_feat))
    pred_512 = pred_flat.reshape(h, w).astype(np.float32)
    pred = F.interpolate(torch.from_numpy(pred_512).unsqueeze(0).unsqueeze(0),
                          size=sample['mask'].shape, mode='nearest').squeeze().long()
    return pred.numpy()


def predict_all_models(sample: Dict[str, np.ndarray]) -> Dict[str, np.ndarray]:
    preds = {}
    for name in MODEL_ORDER:
        if name == 'RandomForest_Classical':
            preds[name] = predict_rf(sample)
        else:
            preds[name] = predict_deep(models_dict[name], sample)
    return preds


print("Prediction functions defined")


Prediction functions defined


## 6. Run Inference on the Test Set (all 5 models)

In [7]:
test_results = []

for info in tqdm(data_split['test'], desc="Test samples"):
    npz_path = Path(info['path'])
    sample = load_sample(npz_path)
    if sample is None:
        print(f"Failed to load: {info['name']}")
        continue

    preds = predict_all_models(sample)
    metrics = {name: evaluate_sample(pred, sample['mask']) for name, pred in preds.items()}

    test_results.append({
        'sample_name': info['name'],
        'm11': sample['m11'],
        'ground_truth': sample['mask'],
        'predictions': preds,
        'metrics': metrics,
    })

print(f"Processed {len(test_results)} / {len(data_split['test'])} test samples across {len(MODEL_ORDER)} models")


Test samples: 100%|██████████| 12/12 [00:24<00:00,  2.06s/it]

Processed 12 / 12 test samples across 5 models


## 7. Per-Sample and Summary Tables

In [8]:
rows = []
for r in test_results:
    for name in MODEL_ORDER:
        m = r['metrics'][name]
        row = {'sample': r['sample_name'], 'model': name,
               'pixel_accuracy': m['pixel_accuracy'], 'mean_tissue_dice': m['mean_tissue_dice']}
        for c, cname in enumerate(Config.CLASS_NAMES):
            for metric_name, val in m['per_class'][c].items():
                row[f'{cname}_{metric_name}'] = val
        rows.append(row)

per_sample_df = pd.DataFrame(rows)
per_sample_df.to_csv(Config.RESULTS_DIR / 'per_sample_all_models.csv', index=False)

summary_rows = []
for name in MODEL_ORDER:
    group = per_sample_df[per_sample_df['model'] == name]
    row = {'Model': MODEL_LABELS[name],
           'Pixel Accuracy': f"{group['pixel_accuracy'].mean():.4f} +/- {group['pixel_accuracy'].std():.4f}",
           'Mean Tissue DSC': f"{group['mean_tissue_dice'].mean():.4f} +/- {group['mean_tissue_dice'].std():.4f}"}
    for cname in Config.CLASS_NAMES:
        row[f'{cname} Dice'] = f"{group[f'{cname}_dice'].mean():.4f} +/- {group[f'{cname}_dice'].std():.4f}"
        row[f'{cname} IoU'] = f"{group[f'{cname}_iou'].mean():.4f} +/- {group[f'{cname}_iou'].std():.4f}"
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(Config.RESULTS_DIR / 'summary_table.csv', index=False)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
summary_df


,Model,Pixel Accuracy,Mean Tissue DSC,Background Dice,Background IoU,Tissue Dice,Tissue IoU,OS Dice,OS IoU,Vaginal Dice,Vaginal IoU
0,"U-Net (pretrained, published)",0.8949 +/- 0.0476,0.7544 +/- 0.1567,0.9271 +/- 0.0937,0.8757 +/- 0.1438,0.8846 +/- 0.0404,0.7952 +/- 0.0647,0.8266 +/- 0.1090,0.7171 +/- 0.1481,0.5521 +/- 0.4133,0.4759 +/- 0.3630
1,U-Net (no pretrained),0.8218 +/- 0.0783,0.6288 +/- 0.1690,0.8937 +/- 0.0997,0.8199 +/- 0.1468,0.8273 +/- 0.0698,0.7106 +/- 0.0947,0.6834 +/- 0.1883,0.5471 +/- 0.2145,0.3757 +/- 0.3194,0.2791 +/- 0.2676
2,U-Net++ (pretrained),0.9086 +/- 0.0439,0.8096 +/- 0.1461,0.9490 +/- 0.0510,0.9068 +/- 0.0865,0.9039 +/- 0.0329,0.8261 +/- 0.0540,0.8442 +/- 0.1315,0.7471 +/- 0.1607,0.6807 +/- 0.3449,0.5935 +/- 0.3285
3,DeepLabV3+ (pretrained),0.9038 +/- 0.0433,0.8011 +/- 0.1324,0.9555 +/- 0.0405,0.9173 +/- 0.0700,0.8930 +/- 0.0437,0.8093 +/- 0.0701,0.8363 +/- 0.1009,0.7292 +/- 0.1333,0.6740 +/- 0.3369,0.5804 +/- 0.3145
4,Random Forest (classical),0.5269 +/- 0.0576,0.3299 +/- 0.0659,0.6088 +/- 0.1084,0.4454 +/- 0.1090,0.5950 +/- 0.0625,0.4260 +/- 0.0623,0.2359 +/- 0.0811,0.1359 +/- 0.0514,0.1586 +/- 0.1473,0.0929 +/- 0.0926


## 8. Best / Median / Worst Sample Identification

Ranked by the **published reference model's** mean tissue Dice, so the same representative cases used in
`m11_unet_test_sample_analysis.ipynb` are reused here -- this notebook then shows how every other method does on
those exact same images.

In [9]:
ranked = sorted(test_results, key=lambda r: r['metrics']['UNet_Pretrained_Published']['mean_tissue_dice'])
n = min(3, len(ranked))
worst_samples = ranked[:n]
median_idx = len(ranked) // 2
median_samples = ranked[median_idx - 1:median_idx + 2] if len(ranked) >= 3 else [ranked[median_idx]]
best_samples = ranked[-n:][::-1]

selected = {'best': best_samples, 'median': median_samples, 'worst': worst_samples}

rows = []
for tier, samples in selected.items():
    for r in samples:
        row = {'tier': tier, 'sample': r['sample_name']}
        for name in MODEL_ORDER:
            row[f'{name}_tissue_dice'] = r['metrics'][name]['mean_tissue_dice']
        rows.append(row)

best_median_worst_df = pd.DataFrame(rows)
best_median_worst_df.to_csv(Config.RESULTS_DIR / 'best_median_worst_samples.csv', index=False)
best_median_worst_df


,tier,sample,UNet_Pretrained_Published_tissue_dice,UNet_NoPretrained_tissue_dice,UNetPlusPlus_Pretrained_tissue_dice,DeepLabV3Plus_Pretrained_tissue_dice,RandomForest_Classical_tissue_dice
0,best,Day0_mm_results_Day0G_8B_S2,0.895270,0.708023,0.881560,0.862116,0.287546
1,best,Day15_mm_results_D15F_S6B_2,0.895026,0.845559,0.917596,0.900126,0.282880
2,best,Day15_mm_results_D15F_S6B_5,0.891771,0.833444,0.913215,0.903109,0.424453
3,median,Day13_mm_results_D13D_S9A_5,0.789751,0.562472,0.796067,0.764884,0.335189
4,median,Day18_mm_results_D18E_S2A_1,0.865333,0.758182,0.890874,0.873224,0.339700
5,median,Day0_mm_results_Day0H_2B_S7,0.871534,0.486963,0.879003,0.839612,0.251801
6,worst,Day12_mm_results_Day12D_9B_S2,0.484805,0.415717,0.448063,0.486609,0.322026
7,worst,Day15_mm_results_D15C_S6A_2,0.540889,0.592828,0.803780,0.833478,0.330649
8,worst,Day6_mm_results_Day6D_8B_S3,0.579294,0.403524,0.924191,0.915267,0.222825


## 9. Qualitative Comparison Figures

One figure per test sample: M11 input, ground truth, and all 5 model predictions, color-coded
(black=Background, blue=Tissue, green=OS, red=Vaginal), with each panel's mean tissue Dice in its title.

In [10]:
def mask_to_rgb(mask: np.ndarray) -> np.ndarray:
    rgb = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for class_id, color in Config.CLASS_COLORS.items():
        rgb[mask == class_id] = color
    return rgb


def plot_comparison_figure(result: dict) -> plt.Figure:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.ravel()

    axes[0].imshow(result['m11'], cmap='gray', vmin=0, vmax=1)
    axes[0].set_title('(A) M11 Image', fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(mask_to_rgb(result['ground_truth']))
    axes[1].set_title('(B) Ground Truth', fontweight='bold')
    axes[1].axis('off')

    for i, name in enumerate(MODEL_ORDER):
        ax = axes[2 + i]
        ax.imshow(mask_to_rgb(result['predictions'][name]))
        dsc = result['metrics'][name]['mean_tissue_dice']
        ax.set_title(f"{MODEL_LABELS[name]}\nTissue DSC: {dsc:.3f}", fontsize=9)
        ax.axis('off')

    axes[-1].axis('off')  # 8th panel unused (7 images total)

    fig.suptitle(f"{result['sample_name']}", fontsize=13, fontweight='bold', y=0.995)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    return fig


pdf_path = Config.RESULTS_DIR / 'all_test_samples_comparison.pdf'
with PdfPages(pdf_path) as pdf:
    for idx, result in enumerate(tqdm(test_results, desc="Rendering comparison figures")):
        fig = plot_comparison_figure(result)

        indiv_pdf = Config.RESULTS_DIR / f"sample_{idx + 1:02d}_{result['sample_name']}_comparison.pdf"
        fig.savefig(indiv_pdf, dpi=300, bbox_inches='tight', format='pdf')

        indiv_tiff = Config.RESULTS_DIR / f"sample_{idx + 1:02d}_{result['sample_name']}_comparison.tiff"
        fig.savefig(indiv_tiff, dpi=300, bbox_inches='tight', format='tiff', pil_kwargs={'compression': 'tiff_lzw'})

        pdf.savefig(fig, dpi=300, bbox_inches='tight')
        plt.close(fig)

print(f"Wrote {len(test_results)} individual PDF/TIFF comparison figures and the combined PDF to:")
print(f"  {Config.RESULTS_DIR}")


Rendering comparison figures: 100%|██████████| 12/12 [01:10<00:00,  5.90s/it]


Wrote 12 individual PDF/TIFF comparison figures and the combined PDF to:
  ..\..\results\model_comparison\test_analysis


## 10. Notes

- All numbers here come from **already-trained** models (`models/best_model.pth` and `models/comparison/*`) -- this
  notebook does no training. Re-run `model_comparison.ipynb` first if you change the training recipe.
- Best/median/worst samples are selected using the **published model** as the anchor, so the qualitative figures
  show the identical cases from `results/analysis_results/` alongside every new candidate's prediction on those
  same images.
- `results/model_comparison/test_analysis/summary_table.csv` is the per-model equivalent of
  `results/analysis_results/table1_overall_performance.csv` / `table2_per_class_performance.csv`.
